In [3]:
# Load Cleaned Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../Data/cleaned_dataset.csv")
print(f"Loaded: {df.shape}")
df.head(3)

Loaded: (3900, 19)


,Customer_ID,Age,Gender,Item_Purchased,Category,Purchase_Amount,Location,Size,Color,Season,Review_Rating,Subscription_Status,Shipping_Type,Discount_Applied,Promo_Code_Used,Previous_Purchases,Payment_Method,Frequency_of_Purchases,Rating_Was_Missing
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,1,Express,1,1,14,Venmo,Fortnightly,0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,1,Express,1,1,2,Cash,Fortnightly,0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,1,Free Shipping,1,1,23,Credit Card,Weekly,0


In [4]:
df.columns.tolist()

['Customer_ID',
 'Age',
 'Gender',
 'Item_Purchased',
 'Category',
 'Purchase_Amount',
 'Location',
 'Size',
 'Color',
 'Season',
 'Review_Rating',
 'Subscription_Status',
 'Shipping_Type',
 'Discount_Applied',
 'Promo_Code_Used',
 'Previous_Purchases',
 'Payment_Method',
 'Frequency_of_Purchases',
 'Rating_Was_Missing']

In [20]:
# Feature 1 — Frequency Score (Numeric)
# Business Question: How often does this customer buy per year?
# Why it matters: High frequency = high engagement = more loyal

freq_map = {
    'Weekly'      : 52,
    'Fortnightly' : 26,
    'Monthly'     : 12,
    'Quarterly'   : 4,
    'Annually'    : 1
}

df['Frequency_Score'] = df['Frequency_of_Purchases'].map(freq_map)

print("Frequency Score distribution:")
print(df['Frequency_Score'].value_counts())
print(f"\nMissing: {df['Frequency_Score'].isnull().sum()}")

Frequency Score distribution:
Frequency_Score
4     1147
26    1089
1      572
12     553
52     539
Name: count, dtype: int64

Missing: 0


Business meaning: A Weekly buyer (score=52) is 52x more engaged than an Annual buyer (score=1). This single conversion unlocks frequency as a numeric variable for scoring and SQL analysis.

In [21]:
# Feature 2 — Promo Dependency Flag

# Business Question: Does this customer NEED a promo to buy?
# Why it matters: Promo-dependent customers are margin destroyers

# Combined check: if EITHER discount or promo was used → promo dependent
df['Promo_Dependent'] = ((df['Discount_Applied'] == 1) | 
                          (df['Promo_Code_Used'] == 1)).astype(int)

print("Promo Dependency:")
print(df['Promo_Dependent'].value_counts())
print(f"\nPromo Dependent %: {df['Promo_Dependent'].mean()*100:.1f}%")

Promo Dependency:
Promo_Dependent
0    2223
1    1677
Name: count, dtype: int64

Promo Dependent %: 43.0%


In [22]:
# Feature 3 — Satisfaction Flag

# Business Question: Is this customer satisfied enough to come back?
# Why it matters: Low satisfaction = churn risk

def satisfaction_flag(rating):
    if rating >= 4.0:
        return 'High'
    elif rating >= 3.0:
        return 'Medium'
    else:
        return 'Low'

df['Satisfaction_Flag'] = df['Review_Rating'].apply(satisfaction_flag)

print("Satisfaction Distribution:")
print(df['Satisfaction_Flag'].value_counts())
print(df['Satisfaction_Flag'].value_counts(normalize=True).round(3) * 100)

Satisfaction Distribution:
Satisfaction_Flag
High      1634
Medium    1586
Low        680
Name: count, dtype: int64
Satisfaction_Flag
High      41.9
Medium    40.7
Low       17.4
Name: proportion, dtype: float64


In [23]:
# Feature 4 — Tenure Band (from Previous Purchases)

# Business Question: How long has this customer been with the brand?
# Why it matters: High previous purchases = established relationship

def tenure_band(prev):
    if prev >= 40:
        return 'Veteran'       # 40–50 purchases
    elif prev >= 25:
        return 'Established'   # 25–39 purchases
    elif prev >= 10:
        return 'Growing'       # 10–24 purchases
    else:
        return 'New'           # 0–9 purchases

df['Tenure_Band'] = df['Previous_Purchases'].apply(tenure_band)

print("Tenure Band distribution:")
print(df['Tenure_Band'].value_counts())

Tenure Band distribution:
Tenure_Band
Growing        1178
Established    1167
Veteran         847
New             708
Name: count, dtype: int64


In [24]:
# Feature 5 — Spend Tier

# Business Question: Is this a high-value or low-value customer by spend?
# Why it matters: High spenders with high frequency = VIP customers

spend_33 = df['Purchase_Amount'].quantile(0.33)
spend_66 = df['Purchase_Amount'].quantile(0.66)

print(f"33rd percentile spend: ${spend_33:.1f}")
print(f"66th percentile spend: ${spend_66:.1f}")

def spend_tier(amount):
    if amount >= spend_66:
        return 'High'
    elif amount >= spend_33:
        return 'Medium'
    else:
        return 'Low'

df['Spend_Tier'] = df['Purchase_Amount'].apply(spend_tier)

print("\nSpend Tier distribution:")
print(df['Spend_Tier'].value_counts())

33rd percentile spend: $45.0
66th percentile spend: $73.0

Spend Tier distribution:
Spend_Tier
High      1348
Medium    1305
Low       1247
Name: count, dtype: int64


In [25]:
# Feature 6 — Customer Value Score (CVS)

# Business Question: Who is our most valuable customer overall?
# Formula: Normalized weighted sum of spend + frequency + previous purchases
# Why it matters: Combines three dimensions into one actionable score

# Normalize each component to 0–1
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df['Norm_Spend']    = scaler.fit_transform(df[['Purchase_Amount']])
df['Norm_Frequency']= scaler.fit_transform(df[['Frequency_Score']])
df['Norm_Previous'] = scaler.fit_transform(df[['Previous_Purchases']])

# Weighted formula: Frequency most important, then previous, then spend
# Weights: Frequency=40%, Previous Purchases=40%, Spend=20%
df['Customer_Value_Score'] = (
    0.40 * df['Norm_Frequency'] +
    0.40 * df['Norm_Previous'] +
    0.20 * df['Norm_Spend']
)

df['Customer_Value_Score'] = df['Customer_Value_Score'].round(4)

print("Customer Value Score Stats:")
print(df['Customer_Value_Score'].describe().round(3))

Customer Value Score Stats:
count    3900.000
mean        0.427
std         0.187
min         0.016
25%         0.289
50%         0.416
75%         0.550
max         0.998
Name: Customer_Value_Score, dtype: float64


In [26]:
#Feature 7 — Value Tier (from Customer Value Score)
# Business Question: Which tier does this customer belong to?
# Why it matters: Enables the Customer Pyramid visualization

def value_tier(score):
    if score >= 0.65:
        return 'Platinum'
    elif score >= 0.45:
        return 'Gold'
    elif score >= 0.25:
        return 'Silver'
    else:
        return 'Bronze'

df['Value_Tier'] = df['Customer_Value_Score'].apply(value_tier)

print("Value Tier distribution:")
print(df['Value_Tier'].value_counts())
print()
print(df.groupby('Value_Tier')['Purchase_Amount'].mean().round(2))

Value Tier distribution:
Value_Tier
Silver      1560
Gold        1152
Bronze       685
Platinum     503
Name: count, dtype: int64

Value_Tier
Bronze      48.30
Gold        65.42
Platinum    69.78
Silver      57.39
Name: Purchase_Amount, dtype: float64


In [27]:
# Feature 8 — Promo + Value Interaction (Key Insight Feature)

# Business Question: Who is high value BUT promo dependent?
# Why it matters: These are customers worth CONVERTING — they buy a lot
# but always with discounts. If we can wean them off promos = margin saved

def promo_value_segment(row):
    if row['Value_Tier'] in ['Platinum', 'Gold'] and row['Promo_Dependent'] == 1:
        return 'High_Value_Promo_Dependent'
    elif row['Value_Tier'] in ['Platinum', 'Gold'] and row['Promo_Dependent'] == 0:
        return 'High_Value_Organic'
    elif row['Value_Tier'] in ['Silver', 'Bronze'] and row['Promo_Dependent'] == 1:
        return 'Low_Value_Promo_Dependent'
    else:
        return 'Low_Value_Organic'

df['Promo_Value_Segment'] = df.apply(promo_value_segment, axis=1)

print("Promo Value Segments:")
print(df['Promo_Value_Segment'].value_counts())

Promo Value Segments:
Promo_Value_Segment
Low_Value_Organic             1293
Low_Value_Promo_Dependent      952
High_Value_Organic             930
High_Value_Promo_Dependent     725
Name: count, dtype: int64


In [28]:
# Feature 9 — Age Group
# Business Question: Which age groups are underlevered?

def age_group(age):
    if age < 25:
        return '18-24'
    elif age < 35:
        return '25-34'
    elif age < 45:
        return '35-44'
    elif age < 55:
        return '45-54'
    elif age < 65:
        return '55-64'
    else:
        return '65+'

df['Age_Group'] = df['Age'].apply(age_group)

print("Age Group distribution:")
print(df['Age_Group'].value_counts().sort_index())

Age Group distribution:
Age_Group
18-24    486
25-34    755
35-44    729
45-54    752
55-64    751
65+      427
Name: count, dtype: int64


In [29]:
# Feature 10 — Premium Shipping Flag

# Business Question: Do high-value customers choose premium shipping?
# Why it matters: Premium shipping choice = willingness to pay more

premium_shipping = ['Express', 'Next Day Air', '2-Day Shipping']

df['Premium_Shipping_Flag'] = df['Shipping_Type'].isin(premium_shipping).astype(int)

print("Premium Shipping Flag:")
print(df['Premium_Shipping_Flag'].value_counts())
print(f"\n% choosing premium: {df['Premium_Shipping_Flag'].mean()*100:.1f}%")

Premium Shipping Flag:
Premium_Shipping_Flag
0    1979
1    1921
Name: count, dtype: int64

% choosing premium: 49.3%


In [30]:
# Feature 11 — High Value Flag (Binary, for SQL filtering)

# Simple binary flag for SQL JOIN and filtering
df['High_Value_Flag'] = (df['Value_Tier'].isin(['Platinum', 'Gold'])).astype(int)

print(f"High Value customers: {df['High_Value_Flag'].sum()} ({df['High_Value_Flag'].mean()*100:.1f}%)")

High Value customers: 1655 (42.4%)


In [31]:
# Feature 12 — Satisfaction × Promo Risk Score

# Business Question: Who is both dissatisfied AND promo-dependent?
# These are your most at-risk customers

sat_map = {'High': 3, 'Medium': 2, 'Low': 1}
df['Sat_Numeric'] = df['Satisfaction_Flag'].map(sat_map)

# Risk = Low Satisfaction + Promo Dependent
# Score 0–4 where 4 = highest risk
df['Retention_Risk_Score'] = (
    (3 - df['Sat_Numeric']) +          # Low satisfaction = higher risk
    df['Promo_Dependent']               # Promo dependent = higher risk
)

print("Retention Risk Score distribution:")
print(df['Retention_Risk_Score'].value_counts().sort_index())

Retention Risk Score distribution:
Retention_Risk_Score
0     942
1    1584
2    1083
3     291
Name: count, dtype: int64


In [32]:
# Save Engineered Dataset

# Drop temporary normalization columns (optional — keep if you want transparency)
cols_to_drop = ['Norm_Spend', 'Norm_Frequency', 'Norm_Previous', 
                'Sat_Numeric', 'Rating_Was_Missing']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

df.to_csv('../Data/engineered_dataset.csv', index=False)
print(f" engineered_dataset.csv saved — Shape: {df.shape}")
print("\nNew columns added:")
new_cols = ['Frequency_Score', 'Promo_Dependent', 'Satisfaction_Flag',
            'Tenure_Band', 'Spend_Tier', 'Customer_Value_Score',
            'Value_Tier', 'Promo_Value_Segment', 'Age_Group',
            'Premium_Shipping_Flag', 'High_Value_Flag', 'Retention_Risk_Score']
print(new_cols)

 engineered_dataset.csv saved — Shape: (3900, 30)

New columns added:
['Frequency_Score', 'Promo_Dependent', 'Satisfaction_Flag', 'Tenure_Band', 'Spend_Tier', 'Customer_Value_Score', 'Value_Tier', 'Promo_Value_Segment', 'Age_Group', 'Premium_Shipping_Flag', 'High_Value_Flag', 'Retention_Risk_Score']
